# Phase 2: Weather Pairing

This notebook converts the Phase 1 dataset manifest into a real image-weather multimodal dataset.

It uses the NASA POWER Daily API to fetch numeric weather for each image using GPS and capture date, then creates same-day, 3-day, 7-day, and 14-day weather features.

Expected input:

```text
DATASET/docs/phase1_manifest.csv
```

Outputs:

```text
DATASET/docs/phase2_outputs/phase2_multimodal_manifest.csv
DATASET/docs/phase2_outputs/multimodal_train.csv
DATASET/docs/phase2_outputs/multimodal_val.csv
DATASET/docs/phase2_outputs/multimodal_test.csv
DATASET/docs/phase2_outputs/weather_cache.json
DATASET/docs/phase2_outputs/phase2_summary.csv
```

NASA POWER Daily API docs: https://power.larc.nasa.gov/docs/services/api/temporal/daily/

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Setup

For local use, keep `PROJECT_ROOT` as the current directory. For Colab, mount Google Drive and set `PROJECT_ROOT` to your project folder.

In [14]:
from pathlib import Path
import json
import hashlib
import collections
import time
from datetime import timedelta

import numpy as np
import pandas as pd
import requests

try:
    from sklearn.model_selection import GroupShuffleSplit
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
    from sklearn.model_selection import GroupShuffleSplit

# Local default. In Colab, change this to your Google Drive project folder.
PROJECT_ROOT = Path("/content/drive/MyDrive")

# Example for Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = Path('/content/drive/MyDrive/PlantDiseaseProject')

DATASET_ROOT = PROJECT_ROOT / "DATASET"
MANIFEST_PATH = DATASET_ROOT / "docs" / "phase1_manifest.csv"
OUTPUT_DIR = DATASET_ROOT / "docs" / "phase2_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Output dir:", OUTPUT_DIR)

Project root: /content/drive/MyDrive
Manifest: /content/drive/MyDrive/DATASET/docs/phase1_manifest.csv
Output dir: /content/drive/MyDrive/DATASET/docs/phase2_outputs


## 2. Load Phase 1 Manifest

In [15]:
def build_phase1_manifest_from_raw_dataset(dataset_root, manifest_path):
    """Rebuild Phase 1 manifest when the CSV was not copied to Colab."""
    metadata_path = dataset_root / "metadata.json"
    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Could not find {manifest_path} or {metadata_path}. "
            "Upload/copy the DATASET folder first, or set PROJECT_ROOT to your Google Drive project folder."
        )

    label_map = {
        "black rot": "Black Rot",
        "esca": "Esca",
        "leaf blight": "Leaf Blight",
        "healthy": "Healthy",
    }

    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    meta_by_file = {item["image_filename"].lower(): item for item in metadata}

    rows = []
    hash_to_paths = collections.defaultdict(list)
    image_dirs = [p for p in dataset_root.iterdir() if p.is_dir() and p.name != "docs"]

    for folder in sorted(image_dirs, key=lambda p: p.name.lower()):
        for image_file in sorted(folder.glob("*.jpg")):
            file_hash = hashlib.md5(image_file.read_bytes()).hexdigest()
            rel_path = image_file.relative_to(dataset_root).as_posix()
            hash_to_paths[file_hash].append(rel_path)

            meta = meta_by_file.get(image_file.name.lower(), {})
            env = meta.get("Environmental_Conditions", {}) if meta else {}
            loc = meta.get("Location", {}) if meta else {}
            gps = loc.get("GPS", {}) if loc else {}
            device = meta.get("Device_Specifications", {}) if meta else {}
            folder_label = label_map.get(folder.name.lower(), folder.name)
            capture_datetime = env.get("Time_of_Capture", "")

            rows.append({
                "image_path": rel_path,
                "filename": image_file.name,
                "label": folder_label,
                "folder": folder.name,
                "metadata_disease_type": meta.get("Disease_Type", ""),
                "metadata_health_status": meta.get("Plant_Health_Status", ""),
                "capture_datetime": capture_datetime,
                "capture_date": capture_datetime[:10] if capture_datetime else "",
                "vineyard": loc.get("Vineyard", ""),
                "latitude": gps.get("Latitude", ""),
                "longitude": gps.get("Longitude", ""),
                "lighting": env.get("Lighting", ""),
                "categorical_weather": env.get("Weather", ""),
                "device_name": device.get("Device_Name", ""),
                "file_hash": file_hash,
                "metadata_found": bool(meta),
            })

    duplicate_group_ids = {
        file_hash: group_id + 1
        for group_id, file_hash in enumerate(sorted(hash_to_paths))
        if len(hash_to_paths[file_hash]) > 1
    }

    for row in rows:
        file_hash = row["file_hash"]
        row["duplicate_group_id"] = duplicate_group_ids.get(file_hash, "")
        row["is_exact_duplicate"] = "yes" if len(hash_to_paths[file_hash]) > 1 else "no"
        row["label_metadata_mismatch"] = (
            "yes"
            if row["metadata_disease_type"] and row["metadata_disease_type"] != row["label"]
            else "no"
        )

    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(manifest_path, index=False)
    print(f"Created missing Phase 1 manifest: {manifest_path}")
    return manifest_path


if not MANIFEST_PATH.exists():
    print(f"Phase 1 manifest not found: {MANIFEST_PATH}")
    print("Trying to rebuild it from DATASET/metadata.json and image folders...")
    build_phase1_manifest_from_raw_dataset(DATASET_ROOT, MANIFEST_PATH)


df = pd.read_csv(MANIFEST_PATH)

required = [
    "image_path", "label", "capture_date", "latitude", "longitude",
    "vineyard", "file_hash", "duplicate_group_id", "is_exact_duplicate"
]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["capture_date"] = pd.to_datetime(df["capture_date"])
df["latitude"] = pd.to_numeric(df["latitude"])
df["longitude"] = pd.to_numeric(df["longitude"])

print("Rows:", len(df))
print("Classes:")
display(df["label"].value_counts().rename_axis("label").reset_index(name="count"))
print("Date range:", df["capture_date"].min().date(), "to", df["capture_date"].max().date())
print("Unique vineyards:", df["vineyard"].nunique())


Rows: 3477
Classes:


,label,count
0,Healthy,1109
1,Esca,888
2,Black Rot,808
3,Leaf Blight,672


Date range: 2023-06-15 to 2023-06-15
Unique vineyards: 28


## 3. Prepare NASA POWER Query Points

NASA POWER daily data is gridded. To avoid unnecessary repeated API calls, this notebook rounds image coordinates to the nearest 0.5-degree grid cell and fetches weather once per grid cell.

In [16]:
def round_to_power_grid(value, resolution=0.5):
    return round(float(value) / resolution) * resolution

df["power_latitude"] = df["latitude"].apply(round_to_power_grid)
df["power_longitude"] = df["longitude"].apply(round_to_power_grid)

min_date = df["capture_date"].min() - timedelta(days=13)
max_date = df["capture_date"].max()

query_points = (
    df[["power_latitude", "power_longitude"]]
    .drop_duplicates()
    .sort_values(["power_latitude", "power_longitude"])
    .reset_index(drop=True)
)

print("NASA POWER query points:", len(query_points))
print("Weather fetch window:", min_date.date(), "to", max_date.date())
display(query_points)

NASA POWER query points: 2
Weather fetch window: 2023-06-02 to 2023-06-15


,power_latitude,power_longitude
0,18.5,74.0
1,19.0,74.0


## 4. Fetch Weather From NASA POWER

Fetched variables:

- `T2M`: daily mean temperature at 2 meters, Celsius
- `RH2M`: daily mean relative humidity at 2 meters, percent
- `PRECTOTCORR`: daily corrected precipitation, mm/day

The API response is cached in `weather_cache.json` so rerunning the notebook does not repeatedly call NASA POWER.

In [17]:
POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
PARAMETERS = "T2M,RH2M,PRECTOTCORR"
CACHE_PATH = OUTPUT_DIR / "weather_cache.json"

if CACHE_PATH.exists():
    weather_cache = json.loads(CACHE_PATH.read_text())
else:
    weather_cache = {}

def cache_key(lat, lon, start, end):
    return f"{lat:.2f}_{lon:.2f}_{start}_{end}"

def fetch_power_daily(lat, lon, start_date, end_date, retries=3, pause=1.0):
    start = pd.Timestamp(start_date).strftime("%Y%m%d")
    end = pd.Timestamp(end_date).strftime("%Y%m%d")
    key = cache_key(lat, lon, start, end)
    if key in weather_cache:
        return weather_cache[key]

    params = {
        "parameters": PARAMETERS,
        "community": "AG",
        "longitude": float(lon),
        "latitude": float(lat),
        "start": start,
        "end": end,
        "format": "JSON",
        "time-standard": "UTC",
    }

    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(POWER_URL, params=params, timeout=60)
            response.raise_for_status()
            payload = response.json()
            weather_cache[key] = payload
            CACHE_PATH.write_text(json.dumps(weather_cache, indent=2))
            time.sleep(pause)
            return payload
        except Exception as exc:
            last_error = exc
            print(f"Attempt {attempt}/{retries} failed for {lat}, {lon}: {exc}")
            time.sleep(pause * attempt)
    raise RuntimeError(f"NASA POWER request failed for {lat}, {lon}") from last_error

weather_frames = []
for row in query_points.itertuples(index=False):
    lat = float(row.power_latitude)
    lon = float(row.power_longitude)
    payload = fetch_power_daily(lat, lon, min_date, max_date)
    params = payload["properties"]["parameter"]
    dates = sorted(params["T2M"].keys())
    weather_frames.append(pd.DataFrame({
        "power_latitude": lat,
        "power_longitude": lon,
        "date": pd.to_datetime(dates, format="%Y%m%d"),
        "temperature_t2m": [params["T2M"][d] for d in dates],
        "humidity_rh2m": [params["RH2M"][d] for d in dates],
        "rainfall_prectotcorr": [params["PRECTOTCORR"][d] for d in dates],
    }))

weather_df = pd.concat(weather_frames, ignore_index=True)
display(weather_df.head())
display(weather_df.describe(include="all"))

,power_latitude,power_longitude,date,temperature_t2m,humidity_rh2m,rainfall_prectotcorr
0,18.5,74.0,2023-06-02,27.74,61.82,1.02
1,18.5,74.0,2023-06-03,27.67,66.96,0.21
2,18.5,74.0,2023-06-04,27.00,72.42,6.47
3,18.5,74.0,2023-06-05,26.64,70.55,1.08
4,18.5,74.0,2023-06-06,28.22,66.71,0.04


,power_latitude,power_longitude,date,temperature_t2m,humidity_rh2m,rainfall_prectotcorr
count,28.000000,28.0,28,28.000000,28.000000,28.000000
mean,18.750000,74.0,2023-06-08 12:00:00,27.810357,67.844643,1.895714
min,18.500000,74.0,2023-06-02 00:00:00,26.640000,59.700000,0.010000
25%,18.500000,74.0,2023-06-05 00:00:00,27.367500,65.075000,0.672500
50%,18.750000,74.0,2023-06-08 12:00:00,27.725000,67.710000,1.195000
75%,19.000000,74.0,2023-06-12 00:00:00,28.405000,71.852500,3.022500
max,19.000000,74.0,2023-06-15 00:00:00,29.200000,74.730000,6.470000
std,0.254588,0.0,NaN,0.665034,4.070003,1.876274


## 5. Build Weather Windows

Temperature and humidity are averaged over each window. Rainfall is summed over each window.

In [18]:
weather_lookup = {
    (row.power_latitude, row.power_longitude, row.date.date()): row
    for row in weather_df.itertuples(index=False)
}

def window_stats(lat, lon, capture_date, days):
    end = pd.Timestamp(capture_date).date()
    dates = [(pd.Timestamp(end) - timedelta(days=i)).date() for i in range(days - 1, -1, -1)]
    records = []
    for day in dates:
        record = weather_lookup.get((lat, lon, day))
        if record is None:
            raise KeyError(f"Missing weather for {lat}, {lon}, {day}")
        records.append(record)
    return {
        f"temp_mean_{days}d": float(np.mean([r.temperature_t2m for r in records])),
        f"humidity_mean_{days}d": float(np.mean([r.humidity_rh2m for r in records])),
        f"rainfall_sum_{days}d": float(np.sum([r.rainfall_prectotcorr for r in records])),
    }

window_rows = []
for row in df.itertuples(index=False):
    item = {}
    for days in [1, 3, 7, 14]:
        item.update(window_stats(row.power_latitude, row.power_longitude, row.capture_date, days))
    window_rows.append(item)

weather_features = pd.DataFrame(window_rows)
paired = pd.concat([df.reset_index(drop=True), weather_features], axis=1)

# Default feature names used by the first multimodal experiments.
paired["temperature"] = paired["temp_mean_7d"]
paired["humidity"] = paired["humidity_mean_7d"]
paired["rainfall"] = paired["rainfall_sum_7d"]
paired["weather_window"] = "7d"

display(paired.head())
display(paired[["temperature", "humidity", "rainfall"]].describe())

,image_path,filename,label,folder,metadata_disease_type,metadata_health_status,capture_datetime,capture_date,vineyard,latitude,...,temp_mean_7d,humidity_mean_7d,rainfall_sum_7d,temp_mean_14d,humidity_mean_14d,rainfall_sum_14d,temperature,humidity,rainfall,weather_window
0,Black rot/black rot001.jpg,black rot001.jpg,Black Rot,Black rot,Black Rot,Diseased,2023-06-15T15:13:00+05:30,2023-06-15,Vineyard_22,18.785238,...,28.012857,68.017143,15.55,28.084286,66.777857,24.94,28.012857,68.017143,15.55,7d
1,Black rot/black rot002.jpg,black rot002.jpg,Black Rot,Black rot,Black Rot,Diseased,2023-06-15T12:58:00+05:30,2023-06-15,Vineyard_28,18.731029,...,27.298571,71.082857,16.57,27.536429,68.911429,28.14,27.298571,71.082857,16.57,7d
2,Black rot/black rot003.jpg,black rot003.jpg,Black Rot,Black rot,Black Rot,Diseased,2023-06-15T12:36:00+05:30,2023-06-15,Vineyard_14,18.802297,...,28.012857,68.017143,15.55,28.084286,66.777857,24.94,28.012857,68.017143,15.55,7d
3,Black rot/black rot004.jpg,black rot004.jpg,Black Rot,Black rot,Black Rot,Diseased,2023-06-15T08:40:00+05:30,2023-06-15,Vineyard_20,18.775820,...,28.012857,68.017143,15.55,28.084286,66.777857,24.94,28.012857,68.017143,15.55,7d
4,Black rot/black rot005.jpg,black rot005.jpg,Black Rot,Black rot,Black Rot,Diseased,2023-06-15T18:02:00+05:30,2023-06-15,Vineyard_8,18.790580,...,28.012857,68.017143,15.55,28.084286,66.777857,24.94,28.012857,68.017143,15.55,7d


,temperature,humidity,rainfall
count,3477.000000,3477.000000,3477.000000
mean,27.890214,68.543525,15.725134
std,0.269410,1.156306,0.384717
min,27.298571,68.017143,15.550000
25%,28.012857,68.017143,15.550000
50%,28.012857,68.017143,15.550000
75%,28.012857,68.017143,15.550000
max,28.012857,71.082857,16.570000


## 6. Create Leakage-Aware Splits

The dataset contains exact duplicate images. This split groups by duplicate hash when duplicates exist, otherwise by vineyard. This prevents exact duplicates from appearing in both training and test splits.

Because all images have the same capture date and nearby GPS values, this is not a perfect real-world generalization split. It is a safer starting point for Colab experiments.

In [19]:
paired["split_group"] = np.where(
    paired["is_exact_duplicate"].astype(str).str.lower().eq("yes"),
    "dup_" + paired["file_hash"].astype(str),
    "vineyard_" + paired["vineyard"].astype(str),
)

seed = 42
groups = paired["split_group"]

train_idx, temp_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed).split(paired, groups=groups))
temp = paired.iloc[temp_idx].reset_index(drop=True)
temp_groups = temp["split_group"]
val_rel_idx, test_rel_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed).split(temp, groups=temp_groups))

paired["split"] = "train"
paired.loc[temp_idx[val_rel_idx], "split"] = "val"
paired.loc[temp_idx[test_rel_idx], "split"] = "test"

split_counts = paired.groupby(["split", "label"]).size().unstack(fill_value=0)
display(split_counts)

leak_check = paired.groupby("split_group")["split"].nunique().max()
print("Max splits per leakage group:", leak_check)
if leak_check != 1:
    raise RuntimeError("Leakage split failed: at least one duplicate/vineyard group appears in multiple splits.")

label,Black Rot,Esca,Healthy,Leaf Blight
split,,,,
test,120,196,238,100
train,576,535,664,470
val,112,157,207,102


Max splits per leakage group: 1


## 7. Save Phase 2 Outputs

In [20]:
output_columns = [
    "image_path", "label", "split", "capture_date", "capture_datetime",
    "vineyard", "latitude", "longitude", "power_latitude", "power_longitude",
    "temperature", "humidity", "rainfall", "weather_window",
    "temp_mean_1d", "humidity_mean_1d", "rainfall_sum_1d",
    "temp_mean_3d", "humidity_mean_3d", "rainfall_sum_3d",
    "temp_mean_7d", "humidity_mean_7d", "rainfall_sum_7d",
    "temp_mean_14d", "humidity_mean_14d", "rainfall_sum_14d",
    "lighting", "categorical_weather", "device_name",
    "file_hash", "duplicate_group_id", "is_exact_duplicate", "split_group",
    "metadata_disease_type", "label_metadata_mismatch",
]

paired_out = paired[output_columns].copy()

full_path = OUTPUT_DIR / "phase2_multimodal_manifest.csv"
paired_out.to_csv(full_path, index=False)

for split in ["train", "val", "test"]:
    split_path = OUTPUT_DIR / f"multimodal_{split}.csv"
    paired_out[paired_out["split"] == split].to_csv(split_path, index=False)
    print(split, len(paired_out[paired_out["split"] == split]), split_path)

summary = pd.DataFrame({
    "metric": [
        "total_rows", "train_rows", "val_rows", "test_rows",
        "unique_classes", "unique_vineyards", "unique_power_grid_points",
        "exact_duplicate_rows", "label_metadata_mismatch_rows",
    ],
    "value": [
        len(paired_out),
        int((paired_out["split"] == "train").sum()),
        int((paired_out["split"] == "val").sum()),
        int((paired_out["split"] == "test").sum()),
        paired_out["label"].nunique(),
        paired_out["vineyard"].nunique(),
        paired_out[["power_latitude", "power_longitude"]].drop_duplicates().shape[0],
        int(paired_out["is_exact_duplicate"].astype(str).str.lower().eq("yes").sum()),
        int(paired_out["label_metadata_mismatch"].astype(str).str.lower().eq("yes").sum()),
    ],
})
summary_path = OUTPUT_DIR / "phase2_summary.csv"
summary.to_csv(summary_path, index=False)

print("Saved:", full_path)
print("Saved:", summary_path)
display(summary)

train 2245 /content/drive/MyDrive/DATASET/docs/phase2_outputs/multimodal_train.csv
val 578 /content/drive/MyDrive/DATASET/docs/phase2_outputs/multimodal_val.csv
test 654 /content/drive/MyDrive/DATASET/docs/phase2_outputs/multimodal_test.csv
Saved: /content/drive/MyDrive/DATASET/docs/phase2_outputs/phase2_multimodal_manifest.csv
Saved: /content/drive/MyDrive/DATASET/docs/phase2_outputs/phase2_summary.csv


,metric,value
0,total_rows,3477
1,train_rows,2245
2,val_rows,578
3,test_rows,654
4,unique_classes,4
5,unique_vineyards,28
6,unique_power_grid_points,2
7,exact_duplicate_rows,1514
8,label_metadata_mismatch_rows,1344


## 8. Phase 2 Notes For The Paper

- Weather was fetched from NASA POWER Daily API using image GPS coordinates and capture date.
- Temperature and humidity are window means; rainfall is a window sum.
- The default multimodal feature set uses a 7-day window.
- Same-day, 3-day, 7-day, and 14-day features are all kept for ablation studies.
- Exact duplicates are grouped during splitting to reduce leakage.
- Limitation: this dataset has only one capture date, so temporal-window claims should be cautious.